# Lesson 1: Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [1]:
import openai
import re
import httpx
import os
from openai import OpenAI

In [2]:
base_url = "http://127.0.0.1:8888/v1" 
model_name ="qwen3-0.6b",
#base_url = "http://127.0.0.1:11434/v1"
#model_name = "gemma3:4b-it-qat"
client = OpenAI(
        api_key="{}".format(os.getenv("API_KEY", "0")),
        base_url= base_url)


In [3]:
chat_completion = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "Hello world"}]
)

In [4]:
chat_completion.choices[0].message.content

'Hello there! It\'s nice to "meet" you. 😊 \n\nIs there anything you\'d like to talk about or anything I can help you with?'

In [5]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model=model_name,
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [6]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [7]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [8]:
abot = Agent(prompt)

In [9]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look the dog's weight using average_dog_weight.
Action: average_dog_weight: Toy Poodle
PAUSE


In [10]:
result = average_dog_weight("Toy Poodle")

In [11]:
result

'a toy poodles average weight is 7 lbs'

In [12]:
next_prompt = "Observation: {}".format(result)

In [13]:
abot(next_prompt)

'Answer: A toy poodle weighs 7 lbs'

In [14]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [15]:
abot = Agent(prompt)

In [16]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to find the average weight of each dog breed and then calculate the combined weight.\nAction: average_dog_weight: Border Collie\nPAUSE\n'

In [17]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [18]:
abot(next_prompt)

'Thought: Now I know the average weight of the Border Collie. I need to find the average weight of a Scottish Terrier and then add the two weights together.\nAction: average_dog_weight: Scottish Terrier\nPAUSE\n'

In [19]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [20]:
abot(next_prompt)

'Thought: I now have the average weights for both breeds. I can calculate the combined weight.\nAction: calculate: 37 + 20\nPAUSE'

In [21]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [22]:
abot(next_prompt)

'Answer: Their combined weight is 57 lbs.'

### Add loop 

In [23]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [24]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [25]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight """
query(question)

Thought: I need to find the average weight of each dog breed and then calculate the combined weight.
Action: average_dog_weight: Border Collie
PAUSE

 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: Now I know the weight of the border collie. I need to find the average weight of a Scottish Terrier and then add that to the border collie's weight.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: I now have the average weights for both dogs. I can calculate the combined weight.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: Their combined weight is 57 lbs.


In [26]:
question = """what is 4+5-7*5 """
query(question)

Thought: I need to evaluate this arithmetic expression. I should use the calculate action to perform the calculation.
Action: calculate: 4+5-7*5
PAUSE

 -- running calculate 4+5-7*5
Observation: -26
Answer: -26


In [27]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight. while calcualting combined weight give a 50% weightage to border collie and 150% weightage to scottish terrier """
query(question)

Thought: I need to calculate the weight of the border collie and the scottish terrier, then apply the specified weightages to determine the combined weight. I will use the calculate action for each dog and then sum the results.
Action: calculate: average_dog_weight: Border Collie
PAUSE
Thought: I need to calculate the weight of the scottish terrier.
Action: calculate: average_dog_weight: Scottish Terrier
PAUSE
Thought: Now I need to sum the two calculated weights and apply the specified weightages.
Action: calculate: 0.5 * [result_border_collie] + 1.5 * [result_scottish_terrier]
PAUSE

 -- running calculate average_dog_weight: Border Collie


SyntaxError: invalid syntax (<string>, line 1)

In [1]:
from openai import OpenAI

# Connect to LM Studio
client = OpenAI(base_url="http://localhost:8888/v1", api_key="lm-studio")

# Define a simple function
def say_hello(name: str) -> str:
    print(f"Hello, {name}!")

# Tell the AI about our function
tools = [
    {
        "type": "function",
        "function": {
            "name": "say_hello",
            "description": "Says hello to someone",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The person's name"
                    }
                },
                "required": ["name"]
            }
        }
    }
]

# Ask the AI to use our function
response = client.chat.completions.create(
    model="qwen3-0.6b",
    messages=[{"role": "user", "content": "Can you say hello to Bob the Builder?"}],
    tools=tools
)

# Get the name the AI wants to use a tool to say hello to
# (Assumes the AI has requested a tool call and that tool call is say_hello)
tool_call = response.choices[0].message.tool_calls[0]
name = eval(tool_call.function.arguments)["name"]

# Actually call the say_hello function
say_hello(name) # Prints: Hello, Bob the Builder!



Hello, Bob the Builder!


In [1]:

import openai
import re
import httpx
import os
from openai import OpenAI
client = OpenAI(
        api_key="{}".format(os.getenv("API_KEY", "0")),
        base_url="http://127.0.0.1:1234/v1")
chat_completion = client.chat.completions.create(
    model="qwen3-0.6b",
    messages=[{"role": "user", "content": "Hello world"}]
)
chat_completion.choices[0].message.content

'<think>\nOkay, the user asked for "Hello world". I need to respond politely. Since they mentioned it in a code context, maybe they\'re learning programming or want to test something. I should acknowledge their request and explain that this is a simple program. Let me make sure the response is friendly and clear.\n</think>\n\nHello world! This is a simple program that prints "Hello world" to the console. If you have any questions, feel free to ask! 😊'

In [2]:
student_custom_functions = [
    {
        'name': 'extract_student_info',
        'description': 'Get the student information from the body of the input text',
        'parameters': {
            'type': 'object',
            'properties': {
                'name': {
                    'type': 'string',
                    'description': 'Name of the person'
                },
                'major': {
                    'type': 'string',
                    'description': 'Major subject.'
                },
                'school': {
                    'type': 'string',
                    'description': 'The university name.'
                },
                'grades': {
                    'type': 'integer',
                    'description': 'GPA of the student.'
                },
                'club': {
                    'type': 'string',
                    'description': 'School club for extracurricular activities. '
                }
                
            }
        }
    }
]

In [3]:
student_1_description = "David Nguyen is a sophomore majoring in computer science at Stanford University. He is Asian American and has a 3.8 GPA. David is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after graduating."

In [4]:
student_2_description="Ravi Patel is a sophomore majoring in computer science at the University of Michigan. He is South Asian Indian American and has a 3.7 GPA. Ravi is an active member of the university's Chess Club and the South Asian Student Association. He hopes to pursue a career in software engineering after graduating."

In [5]:
import json
student_description = [student_1_description,student_2_description]
for i in student_description:
    response = client.chat.completions.create(
        model="qwen3-0.6b",
        messages = [{'role': 'user', 'content': i}],
        functions = student_custom_functions,
        function_call = 'auto'
    )

    # Loading the response as a JSON object
    print(response.choices[0].message)
    json_response = json.loads(response.choices[0].message.function_call.arguments)
    print(json_response)

ChatCompletionMessage(content='Okay, this is a great starting point for a profile of David Nguyen! Here’s a breakdown of what we have and some thoughts on how we could expand on this information:\n\n**Key Information:**\n\n* **Name:** David Nguyen\n* **University:** Stanford University\n* **Year:** Sophomore\n* **Major:** Computer Science\n* **Identity:** Asian American\n* **GPA:** 3.8\n* **Skills/Interests:** Programming, Robotics Club\n* **Career Goals:** Artificial Intelligence\n\n**Potential Expansion & Questions to Consider:**\n\nTo flesh out David\'s profile, we could explore these areas:\n\n* **Specific Programming Languages:** Which languages is he proficient in? (e.g., Python, Java, C++, etc.) This adds a specific detail that strengthens his programming claim.\n* **Robotics Club Role:** What is his involvement in the Robotics Club? (e.g., Team Member, Leader, Event Organizer, Specific Project Involvement?)\n* **AI Interests - Why AI?**  What specifically about AI interests him

AttributeError: 'NoneType' object has no attribute 'arguments'

In [10]:
# function to extract html document from given url
from datetime import datetime
import os
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
def fetch_url(url) -> tuple[str] :
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://google.com"
    }
    response = requests.get(url, headers=headers)
    # request for HTML document of given url
    #response = requests.get(url)
    
    # response will be provided in JSON format
    
    return response.text,response.status_code


In [14]:
def analyze_html(url, question) -> str :
    client = OpenAI(base_url="http://localhost:888/v1", api_key="lm-studio")
    model= "qwen2.5-coder-0.5b-instruct"
    messages = [
        {
            "role": "system",
            "content": "You are a helpful coding  assistant",
        }
    ]
    prompt = """
        Analyse given html text content and answer the questions. If you unable to answer just say I don't know.
        HTML Text : {}
        Question : {}
        """
    html_document,_ = fetch_url(url) 
    soup = BeautifulSoup(html_document, 'html.parser')
    html_text = soup.get_text().strip()
    print(html_text)
    
    print(prompt)
    messages.append({"role": "user", "content": prompt.format(html_text,question)})

    response = client.chat.completions.create(model=model, messages=messages)
    print(response.choices[0].message.content)
    return response.choices[0].message.content
    
    

In [15]:
analyze_html("https://www.entrepreneur.com/business-news/elon-musk-posted-nearly-200-times-on-x-in-24-hours/486712", "Extract date details")




Elon Musk Posted Nearly 200 Times on X In 24 Hours | Entrepreneur



































































 



Skip to content




Menu



Close Menu

 
 

Entrepreneur Landing Page

























      Sign In
          





      Subscribe
          




Search











Entrepreneur Landing Page
























Search





Close Menu

 






      Subscribe to Entrepreneur
          





            Starting a Business
          



            Growing a Business
          



            Leadership
          



            Business News
          



            Science & Technology
          



            Money & Finance
          



            Living
          



            Franchise
          





















 
              Write for Entrepreneur
            




















 
              Bookstore
            









 
              Ask an Expert
            



Tips












 
              White Paper

UnboundLocalError: local variable 'response' referenced before assignment